In [1]:
import os
import shutil
from stardist.models import StarDist2D

2025-11-27 19:01:33.479584: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


In [2]:
# 1. define /path/to/trained/model/folder
model_dir = '../model_weights/stardist/Easy/Easy_100%' # example 

model=StarDist2D(config=None, name=model_dir)

# Export model to TensorFlow's SavedModel format (if not already)
model.export_TF()
print(f"model saved at dir {model.logdir}")

2025-11-27 19:01:34.328053: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2025-11-27 19:01:34.328770: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2025-11-27 19:01:34.334479: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:65:00.0 name: NVIDIA RTX A5000 computeCapability: 8.6
coreClock: 1.695GHz coreCount: 64 deviceMemorySize: 23.67GiB deviceMemoryBandwidth: 715.34GiB/s
2025-11-27 19:01:34.334505: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2025-11-27 19:01:34.336241: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2025-11-27 19:01:34.336298: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2025

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.481453, nms_thresh=0.3.
Instructions for updating:
This function will only be available through the v1 compatibility library as tf.compat.v1.saved_model.utils.build_tensor_info or tf.compat.v1.saved_model.build_tensor_info.
INFO:tensorflow:No assets to save.
INFO:tensorflow:No assets to write.


2025-11-27 19:01:35.358735: I tensorflow/compiler/jit/xla_gpu_device.cc:99] Not creating XLA devices, tf_xla_enable_xla_devices not set
2025-11-27 19:01:35.359062: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:65:00.0 name: NVIDIA RTX A5000 computeCapability: 8.6
coreClock: 1.695GHz coreCount: 64 deviceMemorySize: 23.67GiB deviceMemoryBandwidth: 715.34GiB/s
2025-11-27 19:01:35.359115: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2025-11-27 19:01:35.359142: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2025-11-27 19:01:35.359153: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2025-11-27 19:01:35.359163: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcufft.so.10
20

INFO:tensorflow:SavedModel written to: /tmp/tmpm__cs_42/model/saved_model.pb
model saved at dir ../model_weights/stardist/Easy/Easy_100%


In [3]:
# 2. Unzip TensorFlow's SavedModel 

output_tf_model_zip = os.path.join(model.logdir, 'TF_SavedModel.zip')
output_tf_model = os.path.join(model.logdir, 'TF_SavedModel')

shutil.unpack_archive(output_tf_model_zip, output_tf_model)


In [4]:
# 3.-> onnx convert for use in qupath (e.g., named as 'saved_model_opencv_opset11.pb' below ) 

output_onnx_model = os.path.join(model.logdir, 'ONNX_SavedModel')
os.makedirs(output_onnx_model, exist_ok=True)

# opset was 10 as flag 
!python -m tf2onnx.convert --opset 11 --saved-model {output_tf_model} --output_frozen_graph {os.path.join(output_onnx_model, "saved_model_opencv_opset11.pb")}


2025-11-27 19:01:36.247655: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
/home/guoj5/anaconda3/envs/tfonnx/lib/python3.8/runpy.py:127: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to execution of 'tf2onnx.convert'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))
2025-11-27 19:01:37.153222: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2025-11-27 19:01:37.153927: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2025-11-27 19:01:37.158399: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:65:00.0 name: NVIDIA RTX A5000 computeCapability: 8.6
coreClock: 1.695GHz coreCount: 64 deviceMemorySize: 23.67GiB deviceMemoryBandwidth: 715.34GiB/s
2025-11-27 19:01:37.158420

pb file saved path. Assign this to the qupath groovy script in 'AFM_kidney_cells/qupath_stardist'

In [5]:
print(f"the .pb file will be saved at {output_onnx_model}/saved_model_opencv_opset11.pb")

the .pb file will be saved at ../model_weights/stardist/Easy/Easy_100%/ONNX_SavedModel/saved_model_opencv_opset11.pb
